In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 - Silver Transformation
# MAGIC
# MAGIC This notebook cleans and standardizes the Bronze healthcare tables.
# MAGIC
# MAGIC ## Silver Tables Created
# MAGIC - silver_members
# MAGIC - silver_providers
# MAGIC - silver_labs
# MAGIC - silver_medications
# MAGIC - silver_claims
# MAGIC
# MAGIC The Silver layer removes bad records, standardizes fields, converts data types, and prepares the data for Gold layer feature engineering.

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper,
    when,
    lit,
    to_date,
    regexp_replace,
    months_between,
    floor,
    current_timestamp,
    count,
    sum as spark_sum,
    avg,
    max as spark_max
)

In [0]:
# Unity Catalog settings
CATALOG_NAME = "workspace"
SCHEMA_NAME = "default"

spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"USE SCHEMA {SCHEMA_NAME}")

print(f"Using catalog: {CATALOG_NAME}")
print(f"Using schema: {SCHEMA_NAME}")

In [0]:
bronze_members_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_members"
bronze_providers_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_providers"
bronze_labs_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_labs"
bronze_medications_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_medications"
bronze_claims_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_claims"

silver_members_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_members"
silver_providers_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_providers"
silver_labs_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_labs"
silver_medications_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_medications"
silver_claims_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_claims"

print("Bronze and Silver table paths defined.")

In [0]:
bronze_members_df = spark.table(bronze_members_table)
bronze_providers_df = spark.table(bronze_providers_table)
bronze_labs_df = spark.table(bronze_labs_table)
bronze_medications_df = spark.table(bronze_medications_table)
bronze_claims_df = spark.table(bronze_claims_table)

print("Bronze tables loaded successfully.")

In [0]:
silver_members_df = (
    bronze_members_df
    .select(
        trim(col("member_id")).alias("member_id"),
        trim(col("first_name")).alias("first_name"),
        trim(col("last_name")).alias("last_name"),
        to_date(col("date_of_birth")).alias("date_of_birth"),
        trim(col("gender")).alias("raw_gender"),
        trim(col("race")).alias("race"),
        trim(col("zip_code")).alias("zip_code"),
        to_date(col("enrollment_start_date")).alias("enrollment_start_date"),
        to_date(col("enrollment_end_date")).alias("enrollment_end_date")
    )
    .withColumn(
        "gender",
        when(lower(col("raw_gender")).isin("m", "male"), "Male")
        .when(lower(col("raw_gender")).isin("f", "female"), "Female")
        .otherwise("Unknown")
    )
    .withColumn(
        "age",
        floor(months_between(lit("2024-08-31").cast("date"), col("date_of_birth")) / 12)
    )
    .filter(col("member_id").isNotNull())
    .filter(col("date_of_birth").isNotNull())
    .filter(col("age") >= 0)
    .dropDuplicates(["member_id"])
    .drop("raw_gender")
    .withColumn("silver_processed_timestamp", current_timestamp())
)

display(silver_members_df.limit(10))
print(f"Silver members count: {silver_members_df.count():,}")

In [0]:
(
    silver_members_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(silver_members_table)
)

print(f"Created table: {silver_members_table}")

In [0]:
silver_providers_df = (
    bronze_providers_df
    .select(
        trim(col("provider_id")).alias("provider_id"),
        trim(col("provider_name")).alias("provider_name"),
        trim(col("specialty")).alias("raw_specialty"),
        trim(col("city")).alias("city"),
        upper(trim(col("state"))).alias("raw_state")
    )
    .withColumn(
        "state",
        when(col("raw_state").isin("LA", "LOUISIANA"), "LA")
        .otherwise(col("raw_state"))
    )
    .withColumn(
        "specialty",
        when(lower(col("raw_specialty")).contains("primary"), "Primary Care")
        .when(lower(col("raw_specialty")).contains("family"), "Family Medicine")
        .when(lower(col("raw_specialty")).contains("internal"), "Internal Medicine")
        .when(lower(col("raw_specialty")).contains("endo"), "Endocrinology")
        .when(lower(col("raw_specialty")).contains("cardio"), "Cardiology")
        .when(lower(col("raw_specialty")).contains("neph"), "Nephrology")
        .when(lower(col("raw_specialty")).contains("emergency"), "Emergency Medicine")
        .otherwise("Other")
    )
    .filter(col("provider_id").isNotNull())
    .dropDuplicates(["provider_id"])
    .drop("raw_specialty", "raw_state")
    .withColumn("silver_processed_timestamp", current_timestamp())
)

display(silver_providers_df.limit(10))
print(f"Silver providers count: {silver_providers_df.count():,}")

In [0]:
(
    silver_providers_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(silver_providers_table)
)

print(f"Created table: {silver_providers_table}")

In [0]:
silver_labs_df = (
    bronze_labs_df
    .select(
        trim(col("lab_id")).alias("lab_id"),
        trim(col("member_id")).alias("member_id"),
        to_date(col("lab_date")).alias("lab_date"),
        trim(col("lab_type")).alias("lab_type"),
        col("lab_value").cast("double").alias("lab_value")
    )
    .filter(col("lab_id").isNotNull())
    .filter(col("member_id").isNotNull())
    .filter(col("lab_date").isNotNull())
    .filter(lower(col("lab_type")) == "hba1c")
    .filter(col("lab_value").isNotNull())
    .filter((col("lab_value") >= 3.0) & (col("lab_value") <= 20.0))
    .dropDuplicates(["lab_id"])
    .withColumn("silver_processed_timestamp", current_timestamp())
)

display(silver_labs_df.limit(10))
print(f"Silver labs count: {silver_labs_df.count():,}")

In [0]:
(
    silver_labs_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(silver_labs_table)
)

print(f"Created table: {silver_labs_table}")

In [0]:
silver_medications_df = (
    bronze_medications_df
    .select(
        trim(col("medication_id")).alias("medication_id"),
        trim(col("member_id")).alias("member_id"),
        trim(col("medication_name")).alias("raw_medication_name"),
        to_date(col("fill_date")).alias("fill_date"),
        col("days_supply").cast("int").alias("days_supply"),
        col("adherence_flag").cast("int").alias("adherence_flag")
    )
    .withColumn(
        "medication_name",
        when(lower(col("raw_medication_name")).contains("metformin"), "Metformin")
        .when(lower(col("raw_medication_name")).contains("insulin"), "Insulin")
        .when(lower(col("raw_medication_name")).contains("glipizide"), "Glipizide")
        .when(lower(col("raw_medication_name")).contains("jardiance"), "Jardiance")
        .when(lower(col("raw_medication_name")).contains("ozempic"), "Ozempic")
        .when(lower(col("raw_medication_name")).contains("trulicity"), "Trulicity")
        .when(lower(col("raw_medication_name")).contains("lisinopril"), "Lisinopril")
        .when(lower(col("raw_medication_name")).contains("atorvastatin"), "Atorvastatin")
        .otherwise("Other")
    )
    .filter(col("medication_id").isNotNull())
    .filter(col("member_id").isNotNull())
    .filter(col("fill_date").isNotNull())
    .filter(col("days_supply").isin(30, 60, 90))
    .filter(col("adherence_flag").isin(0, 1))
    .dropDuplicates(["medication_id"])
    .drop("raw_medication_name")
    .withColumn("silver_processed_timestamp", current_timestamp())
)

display(silver_medications_df.limit(10))
print(f"Silver medications count: {silver_medications_df.count():,}")

In [0]:
(
    silver_medications_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(silver_medications_table)
)

print(f"Created table: {silver_medications_table}")

In [0]:
silver_claims_df = (
    bronze_claims_df
    .select(
        trim(col("claim_id")).alias("claim_id"),
        trim(col("member_id")).alias("member_id"),
        trim(col("provider_id")).alias("provider_id"),
        to_date(col("claim_date")).alias("claim_date"),
        upper(trim(col("diagnosis_code"))).alias("diagnosis_code"),
        col("claim_amount").cast("double").alias("claim_amount"),
        trim(col("place_of_service")).alias("raw_place_of_service")
    )
    .withColumn(
        "place_of_service",
        when(lower(col("raw_place_of_service")).contains("office"), "Office")
        .when(lower(col("raw_place_of_service")).contains("hospital"), "Hospital")
        .when(lower(col("raw_place_of_service")).contains("emergency"), "Emergency Room")
        .when(lower(col("raw_place_of_service")).contains("telehealth"), "Telehealth")
        .when(lower(col("raw_place_of_service")).contains("urgent"), "Urgent Care")
        .otherwise("Other")
    )
    .withColumn(
        "has_diabetes_claim",
        when(col("diagnosis_code").startswith("E10") | col("diagnosis_code").startswith("E11"), 1)
        .otherwise(0)
    )
    .withColumn(
        "has_hypertension_claim",
        when(col("diagnosis_code").startswith("I10") | col("diagnosis_code").startswith("I11"), 1)
        .otherwise(0)
    )
    .withColumn(
        "has_ckd_claim",
        when(col("diagnosis_code").startswith("N18"), 1)
        .otherwise(0)
    )
    .filter(col("claim_id").isNotNull())
    .filter(col("member_id").isNotNull())
    .filter(col("claim_date").isNotNull())
    .filter(col("claim_amount").isNotNull())
    .filter(col("claim_amount") >= 0)
    .dropDuplicates(["claim_id"])
    .drop("raw_place_of_service")
    .withColumn("silver_processed_timestamp", current_timestamp())
)

display(silver_claims_df.limit(10))
print(f"Silver claims count: {silver_claims_df.count():,}")

In [0]:
(
    silver_claims_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(silver_claims_table)
)

print(f"Created table: {silver_claims_table}")

In [0]:
spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{SCHEMA_NAME} LIKE 'silver_*'").show(truncate=False)

In [0]:
comparison_data = []

table_pairs = [
    ("members", bronze_members_table, silver_members_table),
    ("providers", bronze_providers_table, silver_providers_table),
    ("labs", bronze_labs_table, silver_labs_table),
    ("medications", bronze_medications_table, silver_medications_table),
    ("claims", bronze_claims_table, silver_claims_table)
]

for dataset_name, bronze_table, silver_table in table_pairs:
    bronze_count = spark.table(bronze_table).count()
    silver_count = spark.table(silver_table).count()
    removed_count = bronze_count - silver_count

    comparison_data.append({
        "dataset_name": dataset_name,
        "bronze_count": bronze_count,
        "silver_count": silver_count,
        "records_removed": removed_count
    })

comparison_df = spark.createDataFrame(comparison_data)

display(comparison_df)

In [0]:
from pyspark.sql.functions import sum as spark_sum

def show_null_counts(table_name):
    df = spark.table(table_name)

    null_counts = df.select([
        spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])

    print(f"Null counts for {table_name}")
    display(null_counts)


show_null_counts(silver_members_table)
show_null_counts(silver_labs_table)
show_null_counts(silver_medications_table)
show_null_counts(silver_claims_table)
show_null_counts(silver_providers_table)

In [0]:
print("Silver Members")
display(spark.table(silver_members_table).limit(10))

print("Silver Labs")
display(spark.table(silver_labs_table).limit(10))

print("Silver Medications")
display(spark.table(silver_medications_table).limit(10))

print("Silver Claims")
display(spark.table(silver_claims_table).limit(10))

print("Silver Providers")
display(spark.table(silver_providers_table).limit(10))

In [0]:
# MAGIC %sql
# MAGIC SELECT
# MAGIC   gender,
# MAGIC   COUNT(*) AS member_count
# MAGIC FROM workspace.default.silver_members
# MAGIC GROUP BY gender
# MAGIC ORDER BY member_count DESC;

In [0]:
# MAGIC %sql
# MAGIC SELECT
# MAGIC   MIN(lab_value) AS min_hba1c,
# MAGIC   MAX(lab_value) AS max_hba1c,
# MAGIC   AVG(lab_value) AS avg_hba1c,
# MAGIC   COUNT(*) AS lab_count
# MAGIC FROM workspace.default.silver_labs;

In [0]:
# MAGIC %sql
# MAGIC SELECT
# MAGIC   SUM(has_diabetes_claim) AS diabetes_claims,
# MAGIC   SUM(has_hypertension_claim) AS hypertension_claims,
# MAGIC   SUM(has_ckd_claim) AS ckd_claims,
# MAGIC   COUNT(*) AS total_claims
# MAGIC FROM workspace.default.silver_claims;